In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path
import os
import shutil
import random
import yaml
import pandas as pd
from collections import Counter, defaultdict
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

BASE_DIR = Path("/content/drive/MyDrive/DIS_PROJESI_007")

ALPHADENT_RAW = BASE_DIR / "data_extracted" / "alphadent" / "AlphaDent"
ALPHADENT_PREPARED = BASE_DIR / "data_prepared" / "alphadent_yolo_seg_2class"

RESULTS_DIR = BASE_DIR / "results" / "alphadent_prepare"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("AlphaDent raw:", ALPHADENT_RAW)
print("AlphaDent prepared:", ALPHADENT_PREPARED)
print("Raw var mı?:", ALPHADENT_RAW.exists())

AlphaDent raw: /content/drive/MyDrive/DIS_PROJESI_007/data_extracted/alphadent/AlphaDent
AlphaDent prepared: /content/drive/MyDrive/DIS_PROJESI_007/data_prepared/alphadent_yolo_seg_2class
Raw var mı?: True


In [3]:
def print_tree(root_path, max_depth=3, max_files_per_folder=10):
    root_path = Path(root_path)

    print("\nKlasör yapısı:", root_path)
    print("=" * 100)

    for current_root, dirs, files in os.walk(root_path):
        current_root = Path(current_root)
        depth = len(current_root.relative_to(root_path).parts)

        if depth > max_depth:
            dirs[:] = []
            continue

        indent = "    " * depth
        print(f"{indent}{current_root.name}/")

        file_indent = "    " * (depth + 1)
        for f in files[:max_files_per_folder]:
            print(f"{file_indent}{f}")

        if len(files) > max_files_per_folder:
            print(f"{file_indent}... {len(files) - max_files_per_folder} dosya daha")

print_tree(ALPHADENT_RAW, max_depth=3)


Klasör yapısı: /content/drive/MyDrive/DIS_PROJESI_007/data_extracted/alphadent/AlphaDent
AlphaDent/
    yolo_seg_train.yaml
    images/
        test/
            test_000.jpg
            test_001.jpg
            test_002.jpg
            test_003.jpg
            test_004.jpg
            test_005.jpg
            test_006.jpg
            test_007.jpg
            test_008.jpg
            test_009.jpg
            ... 125 dosya daha
        train/
            p088_M_29_002.jpg
            p088_M_29_003.jpg
            p089_F_40_001.jpg
            p089_F_40_002.jpg
            p089_F_40_003.jpg
            p090_F_47_001.jpg
            p090_F_47_002.jpg
            p090_F_47_003.jpg
            p091_F_40_001.jpg
            p091_F_40_002.jpg
            ... 1227 dosya daha
        valid/
            p001_F_51_001.jpg
            p001_F_51_002.jpg
            p001_F_51_003.jpg
            p001_F_51_004.jpg
            p015_M_53_001.jpg
            p015_M_53_002.jpg
            p015_M_53_003.

In [4]:
image_train_dir = ALPHADENT_RAW / "images" / "train"
image_valid_dir = ALPHADENT_RAW / "images" / "valid"
image_test_dir = ALPHADENT_RAW / "images" / "test"

label_train_dir = ALPHADENT_RAW / "labels" / "train"
label_valid_dir = ALPHADENT_RAW / "labels" / "valid"

train_images = sorted(list(image_train_dir.glob("*.jpg")))
valid_images = sorted(list(image_valid_dir.glob("*.jpg")))
test_images = sorted(list(image_test_dir.glob("*.jpg")))

train_labels = sorted(list(label_train_dir.glob("*.txt")))
valid_labels = sorted(list(label_valid_dir.glob("*.txt")))

print("Train images:", len(train_images))
print("Train labels:", len(train_labels))

print("Valid images:", len(valid_images))
print("Valid labels:", len(valid_labels))

print("Test images:", len(test_images))
print("Test labels: yok")

Train images: 1237
Train labels: 1237
Valid images: 83
Valid labels: 83
Test images: 135
Test labels: yok


In [5]:
yaml_path = ALPHADENT_RAW / "yolo_seg_train.yaml"

print("YAML var mı?:", yaml_path.exists())
print("YAML yolu:", yaml_path)

if yaml_path.exists():
    print("\nİçerik:")
    print(yaml_path.read_text())

YAML var mı?: True
YAML yolu: /content/drive/MyDrive/DIS_PROJESI_007/data_extracted/alphadent/AlphaDent/yolo_seg_train.yaml

İçerik:
path: ./AlphaDent/
train: images/train
val: images/valid
names:
  0: Abrasion
  1: Filling
  2: Crown
  3: Caries 1 class
  4: Caries 2 class
  5: Caries 3 class
  6: Caries 4 class
  7: Caries 5 class
  8: Caries 6 class



In [6]:
def read_yolo_seg_labels(label_paths):
    class_counter = Counter()
    line_counter = 0
    invalid_lines = []

    for label_path in label_paths:
        lines = label_path.read_text().strip().splitlines()

        for line_idx, line in enumerate(lines):
            line = line.strip()

            if line == "":
                continue

            parts = line.split()

            try:
                cls_id = int(float(parts[0]))
            except:
                invalid_lines.append((label_path, line_idx, line))
                continue

            # YOLO-seg için en az class + 3 nokta yani 1 + 6 sayı beklenir
            if len(parts) < 7:
                invalid_lines.append((label_path, line_idx, line))
                continue

            class_counter[cls_id] += 1
            line_counter += 1

    return class_counter, line_counter, invalid_lines

all_label_paths = train_labels + valid_labels

class_counter, total_lines, invalid_lines = read_yolo_seg_labels(all_label_paths)

print("Toplam label satırı:", total_lines)
print("Geçersiz/kısa satır:", len(invalid_lines))
print("\nSınıf dağılımı:")
for cls_id, count in sorted(class_counter.items()):
    print(cls_id, ":", count)

Toplam label satırı: 12872
Geçersiz/kısa satır: 32

Sınıf dağılımı:
0 : 6347
1 : 2373
2 : 589
3 : 797
4 : 1098
5 : 505
6 : 47
7 : 1059
8 : 57


In [17]:
ALPHADENT_CLASS_NAMES = {
    0: "abrasion",
    1: "filling",
    2: "crown",
    3: "caries_1",
    4: "caries_2",
    5: "caries_3",
    6: "caries_4",
    7: "caries_5",
    8: "caries_6"
}

df_original_counts = pd.DataFrame([
    {
        "class_id": cls_id,
        "class_name": ALPHADENT_CLASS_NAMES.get(cls_id, "unknown"),
        "count": count
    }
    for cls_id, count in sorted(class_counter.items())
])

df_original_counts

,class_id,class_name,count
0,0,abrasion,6347
1,1,filling,2373
2,2,crown,589
3,3,caries_1,797
4,4,caries_2,1098
5,5,caries_3,505
6,6,caries_4,47
7,7,caries_5,1059
8,8,caries_6,57


In [18]:
TARGET_CLASS_MAP = {
    0: 0,  # abrasion -> dis_asinmasi

    3: 1,  # caries_1 -> dis_curugu
    4: 1,  # caries_2 -> dis_curugu
    5: 1,  # caries_3 -> dis_curugu
    6: 1,  # caries_4 -> dis_curugu
    7: 1,  # caries_5 -> dis_curugu
    8: 1   # caries_6 -> dis_curugu
}

NEW_CLASS_NAMES = {
    0: "dis_asinmasi",
    1: "dis_curugu"
}

print("Kullanılacak sınıf dönüşümü:")
for old_cls, new_cls in TARGET_CLASS_MAP.items():
    print(f"AlphaDent {old_cls} ({ALPHADENT_CLASS_NAMES.get(old_cls)}) -> {new_cls} ({NEW_CLASS_NAMES[new_cls]})")

print("\nÇıkarılan sınıflar:")
for old_cls in sorted(set(ALPHADENT_CLASS_NAMES.keys()) - set(TARGET_CLASS_MAP.keys())):
    print(f"AlphaDent {old_cls}: {ALPHADENT_CLASS_NAMES[old_cls]}")

Kullanılacak sınıf dönüşümü:
AlphaDent 0 (abrasion) -> 0 (dis_asinmasi)
AlphaDent 3 (caries_1) -> 1 (dis_curugu)
AlphaDent 4 (caries_2) -> 1 (dis_curugu)
AlphaDent 5 (caries_3) -> 1 (dis_curugu)
AlphaDent 6 (caries_4) -> 1 (dis_curugu)
AlphaDent 7 (caries_5) -> 1 (dis_curugu)
AlphaDent 8 (caries_6) -> 1 (dis_curugu)

Çıkarılan sınıflar:
AlphaDent 1: filling
AlphaDent 2: crown


In [19]:
def convert_label_file_to_2class(src_label_path, dst_label_path):
    lines = src_label_path.read_text().strip().splitlines()
    new_lines = []

    for line in lines:
        line = line.strip()

        if line == "":
            continue

        parts = line.split()

        if len(parts) < 7:
            continue

        try:
            old_cls = int(float(parts[0]))
        except:
            continue

        if old_cls not in TARGET_CLASS_MAP:
            continue

        new_cls = TARGET_CLASS_MAP[old_cls]

        # Polygon koordinatlarını aynen koruyoruz
        coords = parts[1:]

        new_line = " ".join([str(new_cls)] + coords)
        new_lines.append(new_line)

    if len(new_lines) > 0:
        dst_label_path.write_text("\n".join(new_lines))
        return True, len(new_lines)
    else:
        return False, 0

In [20]:
if ALPHADENT_PREPARED.exists():
    shutil.rmtree(ALPHADENT_PREPARED)

for split in ["train", "val"]:
    (ALPHADENT_PREPARED / "images" / split).mkdir(parents=True, exist_ok=True)
    (ALPHADENT_PREPARED / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Hazırlanan klasör:")
print(ALPHADENT_PREPARED)

Hazırlanan klasör:
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/alphadent_yolo_seg_2class


In [21]:
def prepare_alphadent_split(src_img_dir, src_label_dir, split_name):
    src_images = sorted(list(src_img_dir.glob("*.jpg")))

    copied_images = 0
    written_labels = 0
    skipped_no_label = 0
    skipped_no_target = 0
    total_objects = 0

    for img_path in src_images:
        label_path = src_label_dir / (img_path.stem + ".txt")

        if not label_path.exists():
            skipped_no_label += 1
            continue

        dst_img_path = ALPHADENT_PREPARED / "images" / split_name / img_path.name
        dst_label_path = ALPHADENT_PREPARED / "labels" / split_name / label_path.name

        ok, obj_count = convert_label_file_to_2class(label_path, dst_label_path)

        if ok:
            shutil.copy2(img_path, dst_img_path)
            copied_images += 1
            written_labels += 1
            total_objects += obj_count
        else:
            skipped_no_target += 1

    print(f"\n{split_name.upper()}")
    print("Kaynak görüntü:", len(src_images))
    print("Kopyalanan görüntü:", copied_images)
    print("Yazılan label:", written_labels)
    print("Hedef sınıf olmadığı için atlanan:", skipped_no_target)
    print("Label olmadığı için atlanan:", skipped_no_label)
    print("Toplam nesne:", total_objects)

prepare_alphadent_split(image_train_dir, label_train_dir, "train")
prepare_alphadent_split(image_valid_dir, label_valid_dir, "val")


TRAIN
Kaynak görüntü: 1237
Kopyalanan görüntü: 1208
Yazılan label: 1208
Hedef sınıf olmadığı için atlanan: 29
Label olmadığı için atlanan: 0
Toplam nesne: 9244

VAL
Kaynak görüntü: 83
Kopyalanan görüntü: 82
Yazılan label: 82
Hedef sınıf olmadığı için atlanan: 1
Label olmadığı için atlanan: 0
Toplam nesne: 666


In [22]:
def count_split(split):
    img_dir = ALPHADENT_PREPARED / "images" / split
    label_dir = ALPHADENT_PREPARED / "labels" / split

    return len(list(img_dir.glob("*.jpg"))), len(list(label_dir.glob("*.txt")))

rows = []

for split in ["train", "val"]:
    img_count, label_count = count_split(split)
    rows.append({
        "split": split,
        "images": img_count,
        "labels": label_count
    })

df_prepared_summary = pd.DataFrame(rows)
df_prepared_summary

,split,images,labels
0,train,1208,1208
1,val,82,82


In [27]:
new_class_counter = Counter()

for label_path in (ALPHADENT_PREPARED / "labels").rglob("*.txt"):
    lines = label_path.read_text().strip().splitlines()

    for line in lines:
        if line.strip() == "":
            continue

        cls_id = int(float(line.split()[0]))
        new_class_counter[cls_id] += 1

df_new_counts = pd.DataFrame([
    {
        "class_id": cls_id,
        "class_name": NEW_CLASS_NAMES[cls_id],
        "count": count
    }
    for cls_id, count in sorted(new_class_counter.items())
])

df_new_counts

,class_id,class_name,count
0,0,dis_asinmasi,6347
1,1,dis_curugu,3563


In [28]:
data_yaml = f"""
path: {ALPHADENT_PREPARED}
train: images/train
val: images/val

names:
  0: dis_asinmasi
  1: dis_curugu
"""

data_yaml_path = ALPHADENT_PREPARED / "data.yaml"
data_yaml_path.write_text(data_yaml.strip())

print("data.yaml oluşturuldu:")
print(data_yaml_path)
print("\nİçerik:")
print(data_yaml_path.read_text())

data.yaml oluşturuldu:
/content/drive/MyDrive/DIS_PROJESI_007/data_prepared/alphadent_yolo_seg_2class/data.yaml

İçerik:
path: /content/drive/MyDrive/DIS_PROJESI_007/data_prepared/alphadent_yolo_seg_2class
train: images/train
val: images/val

names:
  0: dis_asinmasi
  1: dis_curugu


In [29]:
def show_yolo_seg_sample(split="train"):
    img_dir = ALPHADENT_PREPARED / "images" / split
    label_dir = ALPHADENT_PREPARED / "labels" / split

    img_paths = sorted(list(img_dir.glob("*.jpg")))

    if len(img_paths) == 0:
        print("Görüntü yok:", split)
        return

    img_path = random.choice(img_paths)
    label_path = label_dir / (img_path.stem + ".txt")

    img = Image.open(img_path).convert("RGB")
    img_width, img_height = img.size

    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img)

    if label_path.exists():
        lines = label_path.read_text().strip().splitlines()

        for line in lines:
            parts = line.split()
            cls_id = int(float(parts[0]))
            coords = list(map(float, parts[1:]))

            xs = coords[0::2]
            ys = coords[1::2]

            xs = [x * img_width for x in xs]
            ys = [y * img_height for y in ys]

            if len(xs) >= 3:
                ax.plot(xs + [xs[0]], ys + [ys[0]], linewidth=2)

                ax.text(
                    xs[0],
                    ys[0],
                    NEW_CLASS_NAMES[cls_id],
                    color="yellow",
                    fontsize=10,
                    bbox=dict(facecolor="red", alpha=0.5)
                )

    ax.set_title(f"{split} | {img_path.name}")
    ax.axis("off")
    plt.show()

for split in ["train", "val"]:
    for _ in range(2):
        show_yolo_seg_sample(split)

Output hidden; open in https://colab.research.google.com to view.

In [30]:
df_original_counts.to_csv(RESULTS_DIR / "alphadent_original_class_counts.csv", index=False)
df_prepared_summary.to_csv(RESULTS_DIR / "alphadent_prepared_summary.csv", index=False)
df_new_counts.to_csv(RESULTS_DIR / "alphadent_2class_counts.csv", index=False)

print("Kaydedildi:")
print(RESULTS_DIR)

Kaydedildi:
/content/drive/MyDrive/DIS_PROJESI_007/results/alphadent_prepare
